# Demo 03 - readings from the Event Hub, the streaming source

Job task. Reads whatever the devices have published and appends it to `sensor_readings_bronze`.

The same table is also written by the file path in `demo_05`. Two sources, two checkpoints, one
table. Bronze does not care which protocol a reading arrived over, it cares that it is stored with
enough metadata to trace it back. `source_system` is what makes that visible afterwards.

Event Hubs is not Kafka but it exposes a Kafka endpoint on 9093, so this is the plain `kafka`
source with nothing extra installed.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

for w in ["login", "target_catalog", "secret_scope", "secret_key",
          "eventhub_namespace", "eventhub_name", "consumer_group"]:
    dbutils.widgets.text(w, "")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"])

login          = dbutils.widgets.get("login")
catalog        = dbutils.widgets.get("target_catalog")
secret_scope   = dbutils.widgets.get("secret_scope")
secret_key     = dbutils.widgets.get("secret_key")
namespace      = dbutils.widgets.get("eventhub_namespace")
eventhub_name  = dbutils.widgets.get("eventhub_name")
consumer_group = dbutils.widgets.get("consumer_group")
verbose        = dbutils.widgets.get("verbose") == "true"

assert all([login, catalog, secret_scope, secret_key, namespace, eventhub_name, consumer_group])

demo   = f"{login}_demo_bronze"
target = f"{catalog}.{demo}.sensor_readings_bronze"
ckpt   = f"/Volumes/{catalog}/{demo}/checkpoints/readings"
print(f"{namespace}/{eventhub_name} -> {target}")

In [0]:
conn = dbutils.secrets.get(secret_scope, secret_key)

# the shaded class name is required on DBR, the normal org.apache one does not resolve
jaas = ('kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{conn}";')

kafka_options = {
    "kafka.bootstrap.servers": f"{namespace}.servicebus.windows.net:9093",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism":    "PLAIN",
    "kafka.sasl.jaas.config":  jaas,
    "kafka.group.id":          consumer_group,   # must already exist on the hub
    "subscribe":               eventhub_name,
    "startingOffsets":         "earliest",
    "maxOffsetsPerTrigger":    "10000",
    "failOnDataLoss":          "false",          # 1h retention, old events legitimately expire
}

## Parse and land

The message is parsed into columns, and the message itself is kept in `raw_message`. That second
part matters: if a device starts sending a field this schema does not know about, the parsed column
will not appear, but the value is still in the table and can be recovered later without asking the
device to resend anything it no longer has.

Broker metadata comes along too. Partition and offset are the only way to prove after the fact
where a row came from and whether anything was replayed.

No UDF. `from_json` runs in the JVM, a Python UDF would ship every row to a Python process and back
for parsing Spark already does.

In [0]:
import json

READING_SCHEMA = StructType([
    StructField("device_id",     StringType()),
    StructField("reading_ts",    StringType()),
    StructField("cpu_usage_pct", DoubleType()),
    StructField("ram_usage_pct", DoubleType()),
    StructField("disk_temp_c",   IntegerType()),
    StructField("disk_used_pct", DoubleType()),
    StructField("system_health", StringType()),
    StructField("net_in_mbps",   DoubleType()),
    StructField("producer",      StringType()),
])

stream = (spark.readStream.format("kafka")
          .options(**kafka_options)
          .load()
          .select(
              F.col("value").cast("string").alias("raw_message"),
              F.col("partition").alias("eh_partition"),
              F.col("offset").alias("eh_offset"),
              F.col("timestamp").alias("enqueued_ts"),
          )
          .withColumn("r", F.from_json("raw_message", READING_SCHEMA))
          .select("r.*", "raw_message", "eh_partition", "eh_offset", "enqueued_ts")
          .filter(F.col("producer") == login)      # the hub is shared with the whole group
          .withColumn("source_system", F.lit("eventhub"))
          .withColumn("source_file",   F.lit(None).cast("string"))
          .withColumn("ingestion_ts",  F.current_timestamp())
          .withColumn("load_date",     F.current_date()))

q = (stream.writeStream
     .option("checkpointLocation", ckpt)
     .option("mergeSchema", "true")
     .trigger(availableNow=True)
     .toTable(target))
q.awaitTermination()


def batch_rows(p):
    # depending on the runtime this sits at the top level or only on the source,
    # and it is null on the closing batch that finds nothing left
    d = p if isinstance(p, dict) else json.loads(p.json)
    v = d.get("numInputRows")
    if v is None:
        v = (d.get("sources") or [{}])[0].get("numInputRows")
    return int(v or 0)


new_rows = sum(batch_rows(p) for p in q.recentProgress)
print(f"{target}: +{new_rows} new rows, {spark.table(target).count()} total")

In [0]:
if verbose:
    display(spark.table(target)
            .groupBy("source_system", "device_id")
            .agg(F.count("*").alias("readings"),
                 F.min("reading_ts").alias("first"),
                 F.max("reading_ts").alias("last"))
            .orderBy("source_system", "device_id"))